In [26]:
import pandas as pd
import numpy as np
#FEATURE 1 - TRANSACTION PARSER
df = pd.read_csv("Data set for DADS June.csv")

print("Original shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# DATE CLEANING

df["date"] = pd.to_datetime(
    df["Date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

# AMOUNT CLEANING
df["amount"] = (
    df["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

df["amount"] = pd.to_numeric(df["amount"], errors="coerce")

# TYPE STANDARDISATION

df["type"] = (
    df["Type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        "dr": "debit",
        "debit": "debit",
        "cr": "credit",
        "credit": "credit"
    })
)

# MODE CLEANING

df["Mode"] = df["Mode"].astype(str).str.strip()
df["Mode"] = df["Mode"].replace("", np.nan)

# DUPLICATE REMOVAL

before_duplicates = len(df)

df_clean = df.drop_duplicates().copy()

duplicates_removed = before_duplicates - len(df_clean)

# CHECKING

print("\n" + "=" * 60)
print("FEATURE 1 - TRANSACTION PARSER")
print("=" * 60)

print("Original transactions :", before_duplicates)
print("Duplicates removed    :", duplicates_removed)
print("Clean transactions    :", len(df_clean))
print("Invalid dates         :", df_clean["date"].isna().sum())
print("Invalid amounts       :", df_clean["amount"].isna().sum())
print("Transaction types     :", df_clean["type"].unique())

print("\nData types:")
print(df_clean[["date", "amount", "type"]].dtypes)

Original shape: (1328, 8)

Columns:
['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode', 'Ref']

FEATURE 1 - TRANSACTION PARSER
Original transactions : 1328
Duplicates removed    : 18
Clean transactions    : 1310
Invalid dates         : 0
Invalid amounts       : 0
Transaction types     : ['debit' 'credit']

Data types:
date      datetime64[ns]
amount           float64
type              object
dtype: object


In [27]:
#FEATURE 2 - VENDOR EXTRACTOR
vendor_keywords = {

    "Swiggy": [
        "SWIGGY",
        "BUNDL"
    ],

    "Zomato": [
        "ZOMATO"
    ],

    "Amazon Prime": [
        "AMAZON PRIME",
        "AMZN PRIME"
    ],

    "Amazon": [
        "AMAZON",
        "AMZN"
    ],

    "Blinkit": [
        "BLINKIT",
        "GROFERS",
        "INNOVATIVE RETAIL"
    ],

    "Instamart": [
        "INSTAMART"
    ],

    "Zepto": [
        "ZEPTO"
    ],

    "Flipkart": [
        "FLIPKART",
        "FKART"
    ],

    "Myntra": [
        "MYNTRA"
    ],

    "Nykaa": [
        "NYKAA",
        "FSN E-COMMERCE"
    ],

    "Uber": [
        "UBER"
    ],

    "Ola": [
        "OLA",
        "OLACABS",
        "ANI TECHNOLOGIES",
        "ROPPEN"
    ],

    "Rapido": [
        "RAPIDO"
    ],

    "BMTC": [
        "BMTC"
    ],

    "Starbucks": [
        "STARBUCKS"
    ],

    "Third Wave": [
        "THIRDWAVE",
        "THIRD WAVE",
        "TWC INDIA"
    ],

    "Cafe Coffee Day": [
        "CCD",
        "COFFEE DAY"
    ],

    "Truffles": [
        "TRUFFLES"
    ],

    "Dineout": [
        "DINEOUT"
    ],

    "Empire Restaurant": [
        "EMPIRE"
    ],

    "Meghana Foods": [
        "MEGHANA"
    ],

    "Restaurant": [
        "RESTAURANT",
        "BANGALORE RESTAURANT"
    ],

    "BookMyShow": [
        "BOOKMYSHOW",
        "BMS MOVIE"
    ],

    "Netflix": [
        "NETFLIX"
    ],

    "Spotify": [
        "SPOTIFY"
    ],

    "Disney+ Hotstar": [
        "HOTSTAR",
        "DISNEY",
        "BIGTREE ENTERTAINMENT",
        "STAR INDIA"
    ],

    "JioFiber": [
        "JIOFIBER"
    ],

    "Airtel": [
        "AIRTEL",
        "BHARTI AIRTEL"
    ],

    "Vi": [
        "VI POSTPAID",
        "VODAFONE IDEA",
        "VI-RECHARGE"
    ],

    "Jio": [
        "RELIANCE JIO",
        "JIORECHARGE"
    ],

    "BESCOM": [
        "BESCOM",
        "BANGALORE ELEC"
    ],

    "BWSSB": [
        "BWSSB"
    ],

    "Rent": [
        "LANDLORD"
    ],

    "BigBasket": [
        "BIGBASKET",
        "KIRANAKART"
    ],

    "DMart": [
        "DMART",
        "AVENUE SUPERMARTS"
    ],

    "HP Petrol": [
        "HP PETROL"
    ],

    "BPCL": [
        "BPCL"
    ],

    "Indian Oil": [
        "INDIAN OIL",
        "IOC"
    ],

    "Zerodha": [
        "ZERODHA"
    ],

    "Groww": [
        "GROWW"
    ],

    "Cash Withdrawal": [
        "ATM-WDL"
    ],

    "P2P Transfer": [
        "UPI-AMAN",
        "UPI-PRIYA",
        "UPI-ANKIT",
        "UPI-NEHA",
        "UPI-VIKAS",
        "UPI-KARAN",
        "UPI-SNEHA"
    ],

    "Techcrush Labs": [
        "SALARY",
        "TECHCRUSH LABS"
    ]
}


def extract_vendor(description):
    text = str(description).upper()

    for vendor, keywords in vendor_keywords.items():
        for keyword in keywords:
            if keyword in text:
                return vendor

    return "Uncategorised"


df_clean["vendor_clean"] = df_clean["Description"].apply(extract_vendor)


print("\n" + "=" * 60)
print("FEATURE 2 - VENDOR EXTRACTOR")
print("=" * 60)

print("Unique vendors:", df_clean["vendor_clean"].nunique())

print("\nTop vendors:")
print(df_clean["vendor_clean"].value_counts().head(15))

print("\nUncategorised descriptions:")
print(
    df_clean.loc[
        df_clean["vendor_clean"] == "Uncategorised",
        "Description"
    ].unique()
)


FEATURE 2 - VENDOR EXTRACTOR
Unique vendors: 43

Top vendors:
vendor_clean
Swiggy             223
Zomato             121
Ola                101
Amazon              77
Uber                71
Blinkit             63
Zepto               58
Flipkart            47
Starbucks           42
Rapido              41
BMTC                37
Third Wave          31
Restaurant          29
Cafe Coffee Day     26
BigBasket           24
Name: count, dtype: int64

Uncategorised descriptions:
[]


In [28]:
#FEATURE 3 - CATEGORY TAGGER
category_map = {

    # Food
    "Swiggy": "Food Delivery",
    "Zomato": "Food Delivery",

    # Quick Commerce
    "Blinkit": "Quick Commerce",
    "Instamart": "Quick Commerce",
    "Zepto": "Quick Commerce",

    # E-commerce
    "Amazon": "E-commerce",
    "Flipkart": "E-commerce",
    "Myntra": "E-commerce",
    "Nykaa": "E-commerce",

    # Transport
    "Uber": "Transport",
    "Ola": "Transport",
    "Rapido": "Transport",
    "BMTC": "Transport",

    # Cafe
    "Starbucks": "Cafe",
    "Third Wave": "Cafe",
    "Cafe Coffee Day": "Cafe",

    # Restaurants
    "Truffles": "Restaurants",
    "Dineout": "Restaurants",
    "Empire Restaurant": "Restaurants",
    "Meghana Foods": "Restaurants",
    "Restaurant": "Restaurants",

    # Subscriptions
    "Amazon Prime": "Subscriptions",
    "Netflix": "Subscriptions",
    "Spotify": "Subscriptions",
    "Disney+ Hotstar": "Subscriptions",

    # Utilities
    "JioFiber": "Utilities",
    "Airtel": "Utilities",
    "Vi": "Utilities",
    "Jio": "Utilities",
    "BESCOM": "Utilities",
    "BWSSB": "Utilities",
    "Rent": "Utilities",

    # Groceries
    "BigBasket": "Groceries",
    "DMart": "Groceries",

    # Investments
    "Zerodha": "Investments",
    "Groww": "Investments",

    # Fuel
    "HP Petrol": "Fuel",
    "BPCL": "Fuel",
    "Indian Oil": "Fuel",

    # Entertainment
    "BookMyShow": "Entertainment",

    # Special categories
    "P2P Transfer": "Personal Transfer",
    "Cash Withdrawal": "Cash Withdrawal",

    # Income
    "Techcrush Labs": "Income"
}


df_clean["category"] = (
    df_clean["vendor_clean"]
    .map(category_map)
    .fillna("Uncategorised")
)


print("\n" + "=" * 60)
print("FEATURE 3 - CATEGORY TAGGER")
print("=" * 60)

print(df_clean["category"].value_counts())

print("\nUncategorised transactions:",
      (df_clean["category"] == "Uncategorised").sum())


FEATURE 3 - CATEGORY TAGGER
category
Food Delivery        344
Transport            250
E-commerce           163
Quick Commerce       141
Cafe                  99
Restaurants           73
Utilities             49
Groceries             46
Subscriptions         42
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         11
Income                 6
Name: count, dtype: int64

Uncategorised transactions: 0


In [29]:
#FEATURE 4 - SPENDING OVERVIEW
credits = df_clean.loc[
    df_clean["type"] == "credit", "amount"
].sum()

debits = df_clean.loc[
    df_clean["type"] == "debit", "amount"
].sum()

net_change = credits - debits

if credits != 0:
    savings_rate = (net_change / credits) * 100
else:
    savings_rate = 0

# Spending data
debit_data = df_clean[df_clean["type"] == "debit"].copy()

# Exclude transfers and cash withdrawals from consumption analysis
consumption_data = debit_data[
    ~debit_data["category"].isin(
        ["Personal Transfer", "Cash Withdrawal"]
    )
].copy()


category_spend = (
    consumption_data
    .groupby("category")["amount"]
    .sum()
    .sort_values(ascending=False)
)

vendor_spend = (
    consumption_data
    .groupby("vendor_clean")["amount"]
    .sum()
    .sort_values(ascending=False)
)


print("\n" + "=" * 60)
print("FEATURE 4 - SPENDING OVERVIEW")
print("=" * 60)

print(f"Total Credits : ₹{credits:,.2f}")
print(f"Total Debits  : ₹{debits:,.2f}")
print(f"Net Change    : ₹{net_change:,.2f}")
print(f"Savings Rate  : {savings_rate:.2f}%")
print(f"Transactions  : {len(df_clean)}")
print(f"Unique Vendors: {df_clean['vendor_clean'].nunique()}")

print("\nTOP CATEGORIES")
print("-" * 60)

for category, amount in category_spend.head(10).items():

    percentage = (
        amount / consumption_data["amount"].sum()
    ) * 100

    print(
        f"{category:<20} "
        f"{percentage:>6.2f}%   "
        f"₹{amount:>12,.2f}"
    )


print("\nTOP VENDORS")
print("-" * 60)

for vendor, amount in vendor_spend.head(10).items():

    count = (
        consumption_data["vendor_clean"] == vendor
    ).sum()

    print(
        f"{vendor:<20} "
        f"₹{amount:>12,.2f}   "
        f"{count:>4} transactions"
    )


FEATURE 4 - SPENDING OVERVIEW
Total Credits : ₹509,774.00
Total Debits  : ₹1,678,901.00
Net Change    : ₹-1,169,127.00
Savings Rate  : -229.34%
Transactions  : 1310
Unique Vendors: 43

TOP CATEGORIES
------------------------------------------------------------
E-commerce            36.92%   ₹  593,959.00
Investments           15.43%   ₹  248,160.00
Food Delivery          9.38%   ₹  150,839.00
Utilities              9.32%   ₹  149,914.00
Restaurants            7.32%   ₹  117,737.00
Fuel                   5.55%   ₹   89,303.00
Quick Commerce         4.86%   ₹   78,179.00
Transport              3.57%   ₹   57,474.00
Groceries              3.43%   ₹   55,110.00
Cafe                   1.95%   ₹   31,445.00

TOP VENDORS
------------------------------------------------------------
Amazon               ₹  318,612.00     77 transactions
Zerodha              ₹  210,000.00     14 transactions
Flipkart             ₹  177,510.00     47 transactions
Rent                 ₹  108,000.00      6 transac

In [30]:
# FEATURE 5 - MONTHLY TRENDS & TIME-OF-DAY ANALYSIS

print("=" * 60)
print("FEATURE 5 - MONTHLY TRENDS & TIME-OF-DAY ANALYSIS")
print("=" * 60)

# Make a copy
monthly_data = consumption_data.copy()
# Convert Amount into numeric values

monthly_data["Amount_num"] = (
    monthly_data["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace("Rs", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

monthly_data["Amount_num"] = pd.to_numeric(
    monthly_data["Amount_num"],
    errors="coerce"
)

# Convert Date

monthly_data["Date"] = pd.to_datetime(
    monthly_data["Date"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)

# Create month number
monthly_data["month_num"] = monthly_data["Date"].dt.month

month_names = {
    1: "Jan",
    2: "Feb",
    3: "Mar",
    4: "Apr",
    5: "May",
    6: "Jun"
}
# Convert Time safely

monthly_data["hour"] = pd.to_datetime(
    monthly_data["Time"].astype(str),
    errors="coerce",
    format="mixed"
).dt.hour

# MONTHLY CATEGORY SPENDING

monthly_pivot = pd.pivot_table(
    monthly_data,
    values="Amount_num",
    index="category",
    columns="month_num",
    aggfunc="sum",
    fill_value=0
)

monthly_pivot = monthly_pivot.rename(columns=month_names)

available_months = [
    m for m in ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
    if m in monthly_pivot.columns
]

monthly_pivot = monthly_pivot[available_months]

print("\nMONTHLY SPENDING BY CATEGORY")
print("-" * 60)
print(monthly_pivot.round(2))

# TOTAL MONTHLY SPENDING

monthly_totals = monthly_data.groupby("month_num")["Amount_num"].sum()

print("\nTOTAL MONTHLY SPENDING")
print("-" * 60)

for month_num, amount in monthly_totals.items():
    month_name = month_names.get(month_num, str(month_num))
    print(f"{month_name} : ₹{float(amount):,.2f}")

# TIME-OF-DAY ANALYSIS

time_pivot = pd.pivot_table(
    monthly_data,
    values="Amount_num",
    index="category",
    columns="hour",
    aggfunc="sum",
    fill_value=0
)

print("\nTIME-OF-DAY SPENDING BY CATEGORY")
print("-" * 60)
print(time_pivot.round(2))

FEATURE 5 - MONTHLY TRENDS & TIME-OF-DAY ANALYSIS

MONTHLY SPENDING BY CATEGORY
------------------------------------------------------------
month_num           Jan      Feb       Mar      Apr      May       Jun
category                                                              
Cafe             3690.0   4273.0    5448.0   6564.0   5668.0    5802.0
E-commerce      97134.0  92773.0  103772.0  68098.0  95191.0  136991.0
Entertainment    1263.0      0.0    2418.0   1178.0      0.0    1914.0
Food Delivery   22633.0  23740.0   24803.0  27756.0  25408.0   26499.0
Fuel            30322.0   2079.0   26164.0  18718.0   9138.0    2882.0
Groceries       17649.0   7517.0    6205.0   7538.0   8480.0    7721.0
Investments     38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Quick Commerce  11054.0  16231.0   14110.0  16080.0  13185.0    7519.0
Restaurants     16320.0  21772.0   28313.0   7711.0  22286.0   21335.0
Subscriptions    4256.0   6340.0    7952.0   4335.0   2429.0    4597.0
Transpo

In [31]:
#FEATURE 6 - ANOMALY DETECTION
anomaly_data = df_clean[
    (df_clean["type"] == "debit") &
    (~df_clean["category"].isin(
        ["Personal Transfer", "Cash Withdrawal"]
    ))
].copy()


category_mean = (
    anomaly_data
    .groupby("category")["amount"]
    .transform("mean")
)

category_std = (
    anomaly_data
    .groupby("category")["amount"]
    .transform("std")
)


anomaly_data["z_score"] = (
    anomaly_data["amount"] - category_mean
) / category_std


anomalies = anomaly_data[
    anomaly_data["z_score"] > 2
].sort_values(
    "z_score",
    ascending=False
)


print("\n" + "=" * 60)
print("FEATURE 6 - ANOMALY DETECTION")
print("=" * 60)

print("Total anomalies:", len(anomalies))

print("\nTOP 10 ANOMALIES")
print("-" * 60)

for _, row in anomalies.head(10).iterrows():

    print(
        f"{row['date'].strftime('%d-%b-%Y'):<15} "
        f"{row['vendor_clean']:<20} "
        f"{row['category']:<18} "
        f"₹{row['amount']:>10,.2f} "
        f"z={row['z_score']:.2f}"
    )


FEATURE 6 - ANOMALY DETECTION
Total anomalies: 39

TOP 10 ANOMALIES
------------------------------------------------------------
20-May-2024     Blinkit              Quick Commerce     ₹  1,856.00 z=4.80
08-Apr-2024     Blinkit              Quick Commerce     ₹  1,682.00 z=4.15
26-Jun-2024     Amazon               E-commerce         ₹ 22,008.00 z=3.98
07-Feb-2024     Amazon               E-commerce         ₹ 21,986.00 z=3.98
26-Feb-2024     Restaurant           Restaurants        ₹  8,383.00 z=3.88
05-Apr-2024     Blinkit              Quick Commerce     ₹  1,582.00 z=3.79
22-Jun-2024     Dineout              Restaurants        ₹  7,935.00 z=3.63
31-Mar-2024     Meghana Foods        Restaurants        ₹  7,931.00 z=3.63
05-Mar-2024     Amazon               E-commerce         ₹ 19,917.00 z=3.53
04-Mar-2024     Truffles             Restaurants        ₹  7,441.00 z=3.34


In [32]:
# FEATURE 7 - SPENDING ARCHETYPES

print("=" * 60)
print("FEATURE 7 - SPENDING ARCHETYPES")
print("=" * 60)

# Make a copy of consumption data
archetype_data = consumption_data.copy()

# Convert Amount to numeric

archetype_data["Amount_num"] = (
    archetype_data["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace("Rs", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

archetype_data["Amount_num"] = pd.to_numeric(
    archetype_data["Amount_num"],
    errors="coerce"
)

# Total spending

total_spending = archetype_data["Amount_num"].sum()

category_spending = (
    archetype_data.groupby("category")["Amount_num"]
    .sum()
)

category_percent = (
    category_spending / total_spending * 100
)

# Detect archetypes

archetypes = []

# 1. Foodie
foodie_percent = (
    category_percent.get("Food Delivery", 0)
    + category_percent.get("Restaurants", 0)
    + category_percent.get("Cafe", 0)
)

if foodie_percent > 25:
    archetypes.append(
        f"✓ THE FOODIE - {foodie_percent:.2f}% on Food Delivery + Restaurants + Cafe"
    )

# 2. Quick Commerce Junkie
quick_percent = category_percent.get("Quick Commerce", 0)

if quick_percent > 15:
    archetypes.append(
        f"✓ THE QUICK COMMERCE JUNKIE - {quick_percent:.2f}% on Quick Commerce"
    )

# 3. Shopaholic
ecommerce_percent = category_percent.get("E-commerce", 0)

if ecommerce_percent > 15:
    archetypes.append(
        f"✓ THE SHOPAHOLIC - {ecommerce_percent:.2f}% on E-commerce"
    )

# 4. Investor
investment_percent = category_percent.get("Investments", 0)

if investment_percent > 15:
    archetypes.append(
        f"✓ THE INVESTOR - {investment_percent:.2f}% on Investments"
    )

# 5. Cab Commuter
transport_percent = category_percent.get("Transport", 0)

if transport_percent > 10:
    archetypes.append(
        f"✓ THE CAB COMMUTER - {transport_percent:.2f}% on Transport"
    )

# 6. Subscription Lover
if "vendor" in archetype_data.columns:
    subscription_data = archetype_data[
        archetype_data["category"] == "Subscriptions"
    ]

    subscription_vendors = subscription_data["vendor"].nunique()

    if subscription_vendors >= 4:
        archetypes.append(
            f"✓ THE SUBSCRIPTION LOVER - {subscription_vendors} subscription vendors"
        )
# Late-Night Snacker

archetype_data["hour"] = pd.to_datetime(
    archetype_data["Time"].astype(str),
    errors="coerce",
    format="mixed"
).dt.hour

food_data = archetype_data[
    archetype_data["category"] == "Food Delivery"
]

if len(food_data) > 0:

    late_night_food = food_data[
        (food_data["hour"] >= 21) |
        (food_data["hour"] <= 2)
    ]

    late_night_percent = (
        len(late_night_food) / len(food_data) * 100
    )

    if late_night_percent > 50:
        archetypes.append(
            f"✓ THE LATE-NIGHT SNACKER - {late_night_percent:.2f}% of Food Delivery transactions are late-night"
        )

# Savings Rate

type_clean = archetype_data["Type"].astype(str).str.lower()

credits = archetype_data.loc[
    type_clean == "credit", "Amount_num"
].sum()

debits = archetype_data.loc[
    type_clean == "debit", "Amount_num"
].sum()

if credits > 0:
    savings_rate = ((credits - debits) / credits) * 100
else:
    savings_rate = 0

# 7. YOLO Spender
if savings_rate < 10:
    archetypes.append(
        f"✓ THE YOLO SPENDER - Savings Rate: {savings_rate:.2f}%"
    )

# 8. Disciplined Saver
if savings_rate > 40:
    archetypes.append(
        f"✓ THE DISCIPLINED SAVER - Savings Rate: {savings_rate:.2f}%"
    )
# Print final archetypes

print()

if len(archetypes) > 0:
    for archetype in archetypes:
        print(archetype)
else:
    print("No major spending archetype detected.")

print()
print(f"Savings Rate: {savings_rate:.2f}%")

FEATURE 7 - SPENDING ARCHETYPES

✓ THE SHOPAHOLIC - 36.92% on E-commerce
✓ THE INVESTOR - 15.43% on Investments
✓ THE YOLO SPENDER - Savings Rate: 0.00%

Savings Rate: 0.00%


In [24]:
# FEATURE 8 - FINAL SPENDDNA REPORT

print("=" * 65)
print("                 SPENDDNA - FINAL REPORT")
print("=" * 65)

final_df = df.copy()

# Convert Amount to numeric
final_df["Amount_num"] = (
    final_df["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace("Rs", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

final_df["Amount_num"] = pd.to_numeric(
    final_df["Amount_num"],
    errors="coerce"
)

# Standardize Type
final_df["Type_clean"] = (
    final_df["Type"]
    .astype(str)
    .str.lower()
    .str.strip()
)

# Handle Debit/Credit variations
final_df["Type_clean"] = final_df["Type_clean"].replace({
    "dr": "debit",
    "cr": "credit",
    "debit": "debit",
    "credit": "credit"
})

credits = final_df.loc[
    final_df["Type_clean"] == "credit",
    "Amount_num"
].sum()

debits = final_df.loc[
    final_df["Type_clean"] == "debit",
    "Amount_num"
].sum()

net_change = credits - debits

if credits > 0:
    savings_rate = (net_change / credits) * 100
else:
    savings_rate = 0

print("\nFINANCIAL SUMMARY")
print("-" * 65)

print(f"Total Credits       : ₹{credits:,.2f}")
print(f"Total Debits        : ₹{debits:,.2f}")
print(f"Net Change          : ₹{net_change:,.2f}")
print(f"Savings Rate        : {savings_rate:.2f}%")
print(f"Transactions        : {len(final_df)}")

# SPENDING CATEGORIES FROM consumption_data

category_data = consumption_data.copy()

# Convert Amount
category_data["Amount_num"] = (
    category_data["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace("Rs", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

category_data["Amount_num"] = pd.to_numeric(
    category_data["Amount_num"],
    errors="coerce"
)

category_totals = (
    category_data
    .groupby("category")["Amount_num"]
    .sum()
    .sort_values(ascending=False)
)

# TOP SPENDING CATEGORIES

print("\nTOP SPENDING CATEGORIES")
print("-" * 65)

for category, amount in category_totals.head(5).items():

    percentage = (
        amount / category_totals.sum() * 100
        if category_totals.sum() > 0
        else 0
    )

    print(
        f"{category:<20} "
        f"{percentage:>6.2f}%   "
        f"₹{amount:>12,.2f}"
    )

# SPENDING ARCHETYPES
print("\nSPENDING ARCHETYPES")
print("-" * 65)

total_category_spending = category_totals.sum()

ecommerce_percent = (
    category_totals.get("E-commerce", 0)
    / total_category_spending * 100
)

investment_percent = (
    category_totals.get("Investments", 0)
    / total_category_spending * 100
)

quick_percent = (
    category_totals.get("Quick Commerce", 0)
    / total_category_spending * 100
)

transport_percent = (
    category_totals.get("Transport", 0)
    / total_category_spending * 100
)

foodie_amount = (
    category_totals.get("Food Delivery", 0)
    + category_totals.get("Restaurants", 0)
    + category_totals.get("Cafe", 0)
)

foodie_percent = (
    foodie_amount / total_category_spending * 100
)

if ecommerce_percent > 15:
    print(
        f"✓ THE SHOPAHOLIC - "
        f"{ecommerce_percent:.2f}% on E-commerce"
    )

if investment_percent > 15:
    print(
        f"✓ THE INVESTOR - "
        f"{investment_percent:.2f}% on Investments"
    )

if foodie_percent > 25:
    print(
        f"✓ THE FOODIE - "
        f"{foodie_percent:.2f}% on Food-related spending"
    )

if quick_percent > 15:
    print(
        f"✓ THE QUICK COMMERCE JUNKIE - "
        f"{quick_percent:.2f}% on Quick Commerce"
    )

if transport_percent > 10:
    print(
        f"✓ THE CAB COMMUTER - "
        f"{transport_percent:.2f}% on Transport"
    )

if savings_rate < 10:
    print(
        f"✓ THE YOLO SPENDER - "
        f"Savings Rate: {savings_rate:.2f}%"
    )

if savings_rate > 40:
    print(
        f"✓ THE DISCIPLINED SAVER - "
        f"Savings Rate: {savings_rate:.2f}%"
    )
# KEY OBSERVATIONS
print("\nKEY OBSERVATIONS")
print("-" * 65)

if len(category_totals) > 0:

    top_category = category_totals.index[0]
    top_category_amount = category_totals.iloc[0]

    print(
        f"• Highest spending category: "
        f"{top_category} "
        f"(₹{top_category_amount:,.2f})"
    )

print(f"• Total credits/income: ₹{credits:,.2f}")
print(f"• Total debits/spending: ₹{debits:,.2f}")
print(f"• Overall savings rate: {savings_rate:.2f}%")
print(f"• Number of spending categories: {len(category_totals)}")

print("\n" + "=" * 65)
print("              END OF SPENDDNA REPORT")
print("=" * 65)

                 SPENDDNA - FINAL REPORT

FINANCIAL SUMMARY
-----------------------------------------------------------------
Total Credits       : ₹509,774.00
Total Debits        : ₹1,729,708.00
Net Change          : ₹-1,219,934.00
Savings Rate        : -239.31%
Transactions        : 1328

TOP SPENDING CATEGORIES
-----------------------------------------------------------------
E-commerce            36.92%   ₹  593,959.00
Investments           15.43%   ₹  248,160.00
Food Delivery          9.38%   ₹  150,839.00
Utilities              9.32%   ₹  149,914.00
Restaurants            7.32%   ₹  117,737.00

SPENDING ARCHETYPES
-----------------------------------------------------------------
✓ THE SHOPAHOLIC - 36.92% on E-commerce
✓ THE INVESTOR - 15.43% on Investments
✓ THE YOLO SPENDER - Savings Rate: -239.31%

KEY OBSERVATIONS
-----------------------------------------------------------------
• Highest spending category: E-commerce (₹593,959.00)
• Total credits/income: ₹509,774.00
• Total d